## Clean Data

In [10]:
import re
import pandas as pd

INPUT_FILE = "data.txt"
OUTPUT_FILE = "data.csv"

ADDRESS_CLAIM_PGN = 0x00EE00
ZERO_NAME = "0000000000000000"

pattern = re.compile(
    r"\(\s*([0-9]+(?:\.[0-9]+)?)\s*\)\s+"
    r"(can\d+)\s+"
    r"([0-9A-Fa-f]+)\s+"
    r"\[(\d+)\]\s+"
    r"((?:[0-9A-Fa-f]{2}\s*)+)"
)


def decode_j1939_pgn(can_id_hex: str) -> int:
    can_id = int(can_id_hex, 16)

    pf = (can_id >> 16) & 0xFF
    ps = (can_id >> 8) & 0xFF

    # PDU1: PF < 240, PS is destination address, not part of PGN
    if pf < 240:
        return pf << 8

    # PDU2: PF >= 240, PS is group extension, part of PGN
    return (pf << 8) | ps


def get_source_address(can_id_hex: str) -> int:
    can_id = int(can_id_hex, 16)
    return can_id & 0xFF


def classify_label(pgn: int, data: str) -> str:
    if pgn == ADDRESS_CLAIM_PGN and data == ZERO_NAME:
        return "anomaly"

    if pgn == ADDRESS_CLAIM_PGN:
        return "normal"

    return "ignore"


def main():
    rows = []
    elapsed_time = 0.0

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            match = pattern.match(line)

            if not match:
                print(f"Skipping unmatched line {line_number}: {line}")
                continue

            delta_time, iface, can_id, dlc, data_bytes = match.groups()

            elapsed_time += float(delta_time)

            can_id = can_id.upper()
            data = "".join(data_bytes.split()).upper()
            dlc = int(dlc)

            pgn = decode_j1939_pgn(can_id)
            source_address = get_source_address(can_id)
            label = classify_label(pgn, data)

            rows.append({
                "timestamp": round(elapsed_time, 6),
                "interface": iface,
                "can_id": can_id,
                "dlc": dlc,
                "pgn": f"0x{pgn:06X}",
                "source_address": f"0x{source_address:02X}",
                "data": data,
                "label": label,
            })

    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_FILE, index=False, lineterminator="\n")

    print(f"Saved {len(df)} rows to {OUTPUT_FILE}")

    if len(df) > 0:
        print("\nLabel counts:")
        print(df["label"].value_counts())


if __name__ == "__main__":
    main()

Saved 38514 rows to data.csv

Label counts:
label
ignore     38418
anomaly       58
normal        38
Name: count, dtype: int64


In [12]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib

df = pd.read_csv("data.csv")

# Keep only rows used for training
df = df[df["label"].isin(["normal", "anomaly"])].copy()

if df.empty:
    raise ValueError(
        "No normal/anomaly rows found. Your log may not contain PGN 0x00EE00 Address Claimed messages.")


def extract_features(row):
    source_address = int(str(row["source_address"]).replace("0x", ""), 16)
    data = row["data"]

    is_zero_name = 1 if data == "0000000000000000" else 0

    return pd.Series({
        "source_address": source_address,
        "is_zero_name": is_zero_name,
        "timestamp": row["timestamp"],
    })


features = df.apply(extract_features, axis=1)
labels = df["label"].map({"normal": 0, "anomaly": 1})

model = RandomForestClassifier(random_state=42)
model.fit(features, labels)

joblib.dump(model, "model.joblib")

print("✅ Model trained and saved as model.joblib")
print(df["label"].value_counts())

✅ Model trained and saved as model.joblib
label
anomaly    58
normal     38
Name: count, dtype: int64


## Test the Model

In [5]:
import joblib
import pandas as pd

model = joblib.load("model.joblib")

# Example new data
test_data = pd.DataFrame([{
    "source_address": 0x00,
    "is_zero_name": 1,
    "timestamp": 0.001
}])

prediction = model.predict(test_data)

print("Prediction:", "ANOMALY" if prediction[0] == 1 else "NORMAL")

Prediction: ANOMALY
